In [ ]:
# preparação dos dados
# implementação manual do KNN
# implementação usando sklearn
# comparação dos resultados

In [3]:
pip install scikit-learn

   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------- 8.2/8.2 MB 53.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
   --------------------- ------------------ 19.9/36.5 MB 99.3 MB/s eta 0:00:01
   ---------------------------------------  36.4/36.5 MB 90.5 MB/s eta 0:00:01
   ---------------------------------------- 36.5/36.5 MB 59.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd

dados = pd.read_csv("alzheimer_dataset_preparado.csv")

dados.head()

,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,SleepQuality,...,FunctionalAssessment,MemoryComplaints,BehavioralProblems,ADL,Confusion,Disorientation,PersonalityChanges,DifficultyCompletingTasks,Forgetfulness,Diagnosis
0,73,0,0,2,22.927749,0,13.297218,6.327112,1.347214,9.025679,...,6.518877,0,0,1.725883,0,0,0,1,0,0
1,89,0,0,0,26.827681,0,4.542524,7.619885,0.518767,7.151293,...,7.118696,0,0,2.592424,0,0,0,0,1,0
2,73,0,3,1,17.795882,0,19.555085,7.844988,1.826335,9.673574,...,5.895077,0,0,7.119548,0,1,0,1,0,0
3,74,1,0,1,33.800817,1,12.209266,8.428001,7.435604,8.392554,...,8.965106,0,1,6.481226,0,0,0,0,0,0
4,89,0,0,0,20.716974,0,18.454356,6.310461,0.795498,5.597238,...,6.045039,0,0,0.014691,0,0,1,1,0,0


In [5]:
from sklearn.model_selection import train_test_split

X = dados.drop("Diagnosis", axis=1)
y = dados["Diagnosis"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_treino.shape)
print(X_teste.shape)

(1719, 32)
(430, 32)


In [6]:
X_treino = X_treino.values.tolist()
X_teste = X_teste.values.tolist()

y_treino = y_treino.tolist()
y_teste = y_teste.tolist()

print(type(X_treino))
print(type(X_treino[0]))

<class 'list'>
<class 'list'>


In [7]:
import math

def distancia_euclidiana(ponto1, ponto2):
    soma = 0

    for i in range(len(ponto1)):
        soma += (ponto1[i] - ponto2[i]) ** 2

    return math.sqrt(soma)

In [8]:
print(
    distancia_euclidiana(
        X_treino[0],
        X_treino[1]
    )
)

169.878464187355


In [9]:
import math

# Função responsável por calcular a distância euclidiana
# entre dois pontos (pacientes) do dataset
def distancia_euclidiana(ponto1, ponto2):

    # Variável que armazenará a soma das diferenças ao quadrado
    soma = 0

    # Percorre todos os atributos dos dois pacientes
    for i in range(len(ponto1)):

        # Calcula a diferença entre os atributos,
        # eleva ao quadrado e acumula na variável soma
        soma += (ponto1[i] - ponto2[i]) ** 2

    # Retorna a raiz quadrada da soma
    # que representa a distância euclidiana
    return math.sqrt(soma)

In [10]:
# Teste da função criada

print(
    distancia_euclidiana(
        X_treino[0],  # primeiro paciente
        X_treino[1]   # segundo paciente
    )
)

169.878464187355


In [ ]:
# A distância usada pelo KNN 

# x representa os atributos do primeiro paciente.
# y representa os atributos do segundo paciente.
# Quanto menor a distância, mais parecidos os pacientes são.
# O KNN utiliza essa distância para encontrar os vizinhos mais próximos e decidir a classificação.

In [11]:
# Função que encontra os K vizinhos mais próximos
def obter_vizinhos(X_treino, y_treino, paciente_teste, k):

    # Lista que armazenará as distâncias calculadas
    distancias = []

    # Percorre todos os pacientes da base de treino
    for i in range(len(X_treino)):

        # Calcula a distância entre o paciente de teste
        # e o paciente da base de treino
        distancia = distancia_euclidiana(
            paciente_teste,
            X_treino[i]
        )

        # Guarda a distância e a classe correspondente
        distancias.append(
            (distancia, y_treino[i])
        )

    # Ordena da menor distância para a maior
    distancias.sort(key=lambda x: x[0])

    # Retorna apenas os K primeiros vizinhos
    return distancias[:k]

In [12]:
# Teste da função

vizinhos = obter_vizinhos(
    X_treino,
    y_treino,
    X_teste[0],
    k=5
)

print(vizinhos)

[(44.89252492856195, 0), (45.59875789982789, 1), (48.06389437093441, 0), (48.84134172570138, 0), (49.26499391271762, 0)]


[(44.89, 0),
 (45.59, 1),
 (48.06, 0),
 (48.84, 0),
 (49.26, 0)]

In [13]:
# Função que decide a classe do paciente
# através da votação dos vizinhos
def prever_classe(X_treino, y_treino, paciente_teste, k):

    # Obtém os k vizinhos mais próximos
    vizinhos = obter_vizinhos(
        X_treino,
        y_treino,
        paciente_teste,
        k
    )

    # Contador de votos
    votos = {}

    # Percorre os vizinhos encontrados
    for distancia, classe in vizinhos:

        # Se a classe já existe no dicionário
        if classe in votos:
            votos[classe] += 1

        # Caso contrário cria a classe
        else:
            votos[classe] = 1

    # Retorna a classe com mais votos
    return max(votos, key=votos.get)

In [14]:
classe_prevista = prever_classe(
    X_treino,
    y_treino,
    X_teste[0],
    k=5
)

print("Classe prevista:", classe_prevista)
print("Classe real:", y_teste[0])

Classe prevista: 0
Classe real: 0


desempenho do algoritmo em todos os 430 pacientes do conjunto de teste.

In [15]:
# Contador de acertos
acertos = 0

# Percorre todos os pacientes do conjunto de teste
for i in range(len(X_teste)):

    # Faz a previsão
    previsao = prever_classe(
        X_treino,
        y_treino,
        X_teste[i],
        k=5
    )

    # Verifica se acertou
    if previsao == y_teste[i]:
        acertos += 1

# Calcula a acurácia
acuracia = acertos / len(X_teste)

print("Acertos:", acertos)
print("Total:", len(X_teste))
print("Acurácia:", acuracia)

Acertos: 232
Total: 430
Acurácia: 0.5395348837209303
